In [2]:
import duckdb
import pandas as pd 
from sklearn.metrics import mean_absolute_error
con = duckdb.connect("../data/power.duckdb", read_only=True)
con.sql("SET Timezone = 'UTC'")
df = con.sql("Select * FROM features ORDER BY ts_utc").df()
con.close()

df.shape

(70801, 17)

In [4]:
train = df[df["year"].between(2019,2023)]
valid = df[df["year"] == 2024]
test = df[df["year"] == 2025]
print("train:", len(train), "hours")
print("valid:", len(valid), "hours")
print("test:", len(test), "hours")

train: 43824 hours
valid: 8784 hours
test: 8760 hours


In [ ]:
def evaluate(name, actual, predicted):
    """Scoring one models predicition - avg errir and negative hours that are caught"""
    actual_neg = actual < 0
    predicted_neg = predicted < 0 
    caught = (actual_neg & predicted_neg).sum()
    false_alarms =(~actual_neg & predicted_neg).sum()
    return {
        "model": name,
        "mae_eur_mwh": round(mean_absolute_error(actual, predicted), 1),
        "negative_hours": int(actual_neg.sum()),
        "caught_pct": round(100 * caught / actual_neg.sum(), 1),
        "false_alarms_pct": round(100 * false_alarms / max(predicted_neg.sum(), 1), 1),
        
    }

What it measures:

- mae_eur_mwh: the average error. If the real prices were 50, 80, 100 and the guesses 60, 70, 100, the errors are 10, 10 and 0, so MAE = 6.7 €/MWh. Lower is better.
- caught_pct: of the hours that really were negative, how many did the model also predict as negative? ( call this recall.)
- false_alarm_pct: of the hours the model said would be negative, how many weren’t? (This is 100% minus precision.)
- & means “and”, ~ means “not”. So ~actual_neg & predicted_neg = “not actually negative, but predicted negative” = a false alarm.
- max(predicted_neg.sum(), 1) avoids dividing by zero if a model never predicts a negative hour.

In [7]:
results = pd.DataFrame([
    evaluate("same as yesterday", valid["price_euro_mwh"], valid["price_lag_24h"]),
    evaluate("same as last week", valid["price_euro_mwh"], valid["price_lag_168h"]),
])
results

NameError: name 'predicted_neg_sum' is not defined